# Classification avec pytorch RNN,LSTM,GRU

- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS
- creation date : 2025-11-07

1️⃣ Modèles séquentiels / temporels (RNN, LSTM, GRU)

- Les courbes de lumière sont naturellement des séries temporelles. Au lieu de résumer tout en features, tu peux traiter la séquence complète par bande :
- Input : MJD, flux, flux_err par bande
- RNN / LSTM / GRU → apprend la dynamique des variations de flux dans le temps
- Output : type de SN ou probabilité multi-classes
- 💡 Avantages : capture les formes des courbes (rise, plateau, décroissance)
- 💡 Inconvénients : plus gourmand en données et GPU, nécessite padding ou masking si les séquences ont des longueurs différentes

In [ ]:
import torch.nn as nn

class SN_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        _, (hn, _) = self.lstm(x)  # hn: hidden state final
        out = self.fc(hn[-1])
        return out

- input_dim = 2 ou 3 (flux, flux_err, optionnellement bande encodée)
- hidden_dim : 32–128 suffisent pour commencer
- Prétraitement : séquences padding ou découpées à longueur fixe

2️⃣ Transformers / LLM pour séries temporelles
- Les transformers (ou petits LLMs) commencent à être utilisés pour les lightcurves :
- Input : séquence de tokens ou vecteurs (flux, MJD, bande)
- Attention apprend à se concentrer sur les points clés (pic, montée, décroissance)
- Output : classification multi-classes
- Avantages : meilleure capture des relations longues distances dans la série.
- Inconvénients : lourd, nécessite GPU, tuning complexe.
- Des implémentations existent sur Hugging Face :
- Modèles Time Series Transformer
- AstroNet et SuperNNova pour transients

3️⃣ Approche hybride
- Tu peux combiner features classiques + séquence complète :
- Extraire features globales (flux_max, flux_mean, rise_time)
- Coupler à une LSTM ou transformer pour capturer la dynamique
- Fusionner dans un réseau final (MLP) pour la classification

💡 Suggestion pour ton notebook NERSC :

- Commencer avec MLP sur features (déjà fait)
- Ensuite, préparer les séquences : MJD/flux par bande, padding, mask
- Tester LSTM/GRU sur un sous-échantillon pour vérifier performance et temps GPU
- Enfin, explorer Transformers si tu veux vraiment capturer les patterns complexes

## Prepare data

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from astropy.io import fits
import glob, os
import random
import fitsio

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import sys
print(sys.executable)

In [ ]:
#!ls /global/cfs/cdirs/lsst/www/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2

## Function definitions

In [ ]:
def buildDictObjTypeFromFile(path_top,objtypelist):
    """
    Retrun a dictionnary with the type of object as key and values are dictionary of ELASTIC2 list of header file and phtometric files
    """

    dict_objtype_to_files = {}

    # loop on object type
    for obj_type in objtypelist:
        path = os.path.join(path_top, obj_type)
   
        head_files = sorted(glob.glob(os.path.join(path, "*_HEAD.FITS.gz")))
        phot_files = sorted(glob.glob(os.path.join(path, "*_PHOT.FITS.gz")))
        dict_objtype_to_files[obj_type] = {"head_files" : head_files, "phot_files": phot_files }
    return dict_objtype_to_files

In [ ]:
def load_lightcurve(head_file, phot_file, lctype):
    import pandas as pd
    from astropy.io import fits

    head = fits.open(head_file)[1].data
    phot = fits.open(phot_file)[1].data

    # Récupère l’identifiant SNID depuis le HEAD uniquement
    snid = str(head["SNID"][0]).strip()

    # Création du DataFrame photométrique
    df = pd.DataFrame({
        "SNID": snid,
        "LCTYPE": lctype,
        "MJD": phot["MJD"],
        "BAND": [b.strip() for b in phot["BAND"]],
        "FLUXCAL": phot["FLUXCAL"],
        "FLUXCALERR": phot["FLUXCALERR"]
    })

    # Forcer le little-endian pour toutes les colonnes float ou int
    for col in ["MJD", "FLUXCAL", "FLUXCALERR"]:
        df[col] = df[col].astype(df[col].dtype.newbyteorder('<'))

    return df, head

In [ ]:
def load_all_lightcurves(nf, dict_objtype_tofile):
    """
    """
  
    lightcurves = []
    for obj_type, filesdict in dict_objtype_tofile.items() :
        head_files = filesdict["head_files"]
        phot_files = filesdict["phot_files"]
        for h, p in zip(head_files[:nf], phot_files[:nf]):
            try:
                df, head = load_lightcurve(h, p, obj_type)
                lightcurves.append((df, head))
            except Exception as e:
                print(f"Erreur avec {h}: {e}")
    return lightcurves



In [ ]:
def plot_lightcurve(df, title=None):
    plt.figure(figsize=(10, 5))
    bands = sorted(df['BAND'].unique())
    for band in bands:
        if band.strip() == '-':  # ignorer les bandes bidon
            continue
        d = df[df['BAND'] == band]
        plt.errorbar(d['MJD'], d['FLUXCAL'], yerr=d['FLUXCALERR'], fmt='o', label=band.strip())
    plt.xlabel("MJD")
    plt.ylabel("Flux")
    if title:
        plt.title(title)
    plt.legend()
    plt.show()

## Start 

In [ ]:
# Répertoire d'un type de SN
BASE_DIR = "/global/cfs/cdirs/lsst/www/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
sn_type = "ELASTICC2_TRAIN_02_SNIa-SALT3"
sample_types = [
    "ELASTICC2_TRAIN_02_SNIa-SALT3",
    "ELASTICC2_TRAIN_02_SNIc-Templates",
    "ELASTICC2_TRAIN_02_SNIb-Templates"]


dict_sntype_to_files = buildDictObjTypeFromFile(path_top = BASE_DIR,objtypelist = sample_types )

In [ ]:
# -----------------------------
# 3️⃣ Lire un HEAD aléatoire
# -----------------------------
# select the first type of SN
sntype = list(dict_sntype_to_files.keys())[0]

#then specify the path
head_files = dict_sntype_to_files[sntype]["head_files"]

head_file = os.path.join(PATH, random.choice(head_files))
head_data = fitsio.FITS(head_file)[1].read()
print("Colonnes HEAD :", head_data.dtype.names)
print("Exemple HEAD :\n", head_data[:1])

### Load a number of SN samples

In [ ]:
NF= 50

lightcurves = load_all_lightcurves(NF, dict_sntype_to_files)

print(f"{len(lightcurves)} curves loaded")

In [ ]:
lightcurves[0]

## Plot First Light Curve

In [ ]:
index = 1
plot_lightcurve(lightcurves[index][0], title="SNID " + str(lightcurves[index][1]['SNID']))

In [ ]:
assert False

In [ ]:
import numpy as np

def extract_features(df):
    bands = ['u', 'g', 'r', 'i', 'z', 'Y']
    features = []
    for band in bands:
        d = df[df['BAND'].str.strip() == band]
        if len(d) == 0:
            features.extend([np.nan, np.nan, np.nan, np.nan])
        else:
            flux = d['FLUXCAL'].values
            mjd = d['MJD'].values
            features.extend([
                np.max(flux),
                mjd[np.argmax(flux)],
                np.mean(flux),
                np.std(flux)
            ])
    return np.array(features)


In [ ]:
lc = lightcurves[0]

In [ ]:
lc[0]['LCTYPE'].unique()[0]

In [ ]:
X = np.array([extract_features(lc[0]) for lc in lightcurves])
y = np.array([lc[0]['LCTYPE'].unique()[0] for lc in lightcurves])  # ou le label que tu as

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))


## Classify with pytorch

- Super 😄 ! On va passer à une version PyTorch pour classifier les transients ELAsTiCC2 à partir des features extraites.
- L’idée :
- Dataset X et labels y (features comme flux_max, flux_mean, rise_time, etc.)
- Encodage des labels en indices (SNIa-SALT3 → 0, SNIc → 1, …)
- Mini réseau fully-connected (MLP)
- Entraînement et évaluation avec PyTorch

🔹 Ce que ce notebook PyTorch fait :
- Encode les labels en indices numériques (LabelEncoder).
- Utilise MLP simple avec 2 couches cachées.
- Optimiseur Adam, loss CrossEntropy.
- Mini-batch pour un entraînement efficace même sur NERSC CPU ou GPU.
- Affiche rapport de classification final avec précision par type de SN.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---------------------------
# 1️⃣ Préparer les données
# ---------------------------
X_values = X.astype(np.float32)
le = LabelEncoder()
y_values = le.fit_transform(y)
num_classes = len(le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X_values, y_values, test_size=0.3, random_state=42, stratify=y_values
)

# Convertir en tensors
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# ---------------------------
# 2️⃣ Définir le modèle
# ---------------------------
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)

input_dim = X_train.shape[1]
hidden_dim = 64
output_dim = num_classes

model = MLPClassifier(input_dim, hidden_dim, output_dim)

# ---------------------------
# 3️⃣ Définir loss + optimizer
# ---------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ---------------------------
# 4️⃣ Entraînement
# ---------------------------
epochs = 50
batch_size = 32

for epoch in range(epochs):
    permutation = torch.randperm(X_train_t.size()[0])
    epoch_loss = 0
    
    for i in range(0, X_train_t.size()[0], batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = X_train_t[indices], y_train_t[indices]
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    if (epoch+1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")

# ---------------------------
# 5️⃣ Évaluation
# ---------------------------
model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    y_pred = torch.argmax(logits, dim=1).numpy()

print("Classification report :\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))
